In [ ]:
# Insert the data
file_path <- "/kaggle/input/crimes-in-chicago/Chicago_Crimes_2012_to_2017.csv"
crime_raw <- read.csv(file_path)

In [ ]:
# All libraries used

install.packages("moments")
library(moments)
library(dplyr)
library(tidyr)
library(tidyverse)
library(lubridate) # Required for date() function
library(sf)
library(ggplot2)
library(ggpubr)
library(forecast)

# **Filtering and Cleaning the data**

In [ ]:
################################################
# Cleaning the data
################################################

# Clean Column Names
names(crime_raw) <- tolower(gsub(" ", "_", names(crime_raw)))

# Remove rows with missing locatio, remove data on 2017
crime_clean <- subset(crime_raw, !is.na(latitude) & !is.na(longitude))
crime_clean <- subset(crime_clean, year != "2017")

# Convert Date Column
# Format matches "01/01/2026 11:40:00 PM"
crime_clean$date_time <- as.POSIXct(crime_clean$date, format = "%m/%d/%Y %I:%M:%S %p")

# Create extra time columns (Helps later)
crime_clean$year    <- format(crime_clean$date_time, "%Y")
crime_clean$month   <- format(crime_clean$date_time, "%b") # Abbreviated month
crime_clean$hour    <- format(crime_clean$date_time, "%H") # Hour (00-23)
crime_clean$weekday <- format(crime_clean$date_time, "%a") # Abbreviated weekday

# Check the result
print(paste("Successfully loaded and cleaned", nrow(crime_clean), "rows."))
head(crime_clean)

In [ ]:
################################################
# Normality check
################################################

# We count how many crimes happened on each specific date
daily_counts_table <- table(as.Date(crime_clean$date_time))
crime_counts <- as.numeric(daily_counts_table)

# Q-Q Plot
qqnorm(crime_counts, main = "Normal Q-Q Plot")
qqline(crime_counts, col = "red", lwd = 2)

# Shapiro-Wilk
set.seed(123)
sub_sample <- sample(crime_counts, 100)
shapiro_result <- shapiro.test(sub_sample)

print(paste("Shapiro-Wilk p-value:", shapiro_result$p.value))


In [ ]:
# Calculate Skewness
skew <- skewness(crime_counts)
print(paste("Skewness:", skew))

# Calculate Kurtosis
kurt <- kurtosis(crime_counts)
print(paste("Kurtosis:", kurt))

# **1. Identify the major temporal trends**

In [ ]:
# Temporal Visualization


# matrix: rows = Weekdays ; cols = Hours
counts_matrix <- table(crime_clean$weekday, crime_clean$hour)

# Create a gradient from White to Dark Red
# 100 represents the number of distinct color shades
red_colors <- colorRampPalette(c("white", "#b30000"))(100)

# Plot Heatmap
image(x = 0:23,                  # Hours
      y = 1:7,                   # Weekdays
      z = t(counts_matrix),      # Transposed matrix [Hour, Day]
      col = red_colors,          # Our color palette
      axes = FALSE,              # Turn off default axes
      xlab = "Hours",
      ylab = "Weekdays",
      main = "When does Crime Happen along the Week?")


axis(side = 1, at = seq(0, 23, by = 2))
axis(side = 2, at = 1:7, labels = rownames(counts_matrix), las = 1)
box()

In [ ]:
# t-test to check the "midnight-8 am" significant reduction in criminal activity


# Ensure 'hour' is numeric and verify the full dataset is loaded
crime_clean$hour <- as.numeric(crime_clean$hour)

# Check if you actually have data for all 24 hours
print(table(crime_clean$hour)) 
# ^ If this only shows 0-7, you need to reload your original full dataset!

# Create the Time Block category again
crime_hourly <- crime_clean %>%
  filter(!is.na(hour)) %>% # Remove any missing hours
  mutate(time_block = ifelse(hour >= 0 & hour < 8, "Midnight-8AM", "Rest of Day")) %>%
  group_by(date, time_block) %>%
  summarise(count = n(), .groups = "drop") # groups="drop" prevents grouping errors

# Verify that we have two groups now
print(table(crime_hourly$time_block))

# Run the t-test (only if both groups exist)
if(length(unique(crime_hourly$time_block)) == 2) {
  t_test_result <- t.test(count ~ time_block, data = crime_hourly, alternative = "less")
  print(t_test_result)
} else {
  print("Error: Still missing data for one of the time blocks.")
}

In [ ]:
# One-Way ANOVA to confirm the differences in daily crime means


# Calculate daily total crimes
daily_counts <- crime_clean %>%
  count(date = date(date_time), weekday)

# One-Way ANOVA
anova_model <- aov(n ~ weekday, data = daily_counts)

# View the Results
summary(anova_model)

# Post-hoc Test
# If p < 0.05, use TukeyHSD to see WHICH days are different
TukeyHSD(anova_model)

# **2. Detect the most persistent crime hotspots**

In [ ]:
#Plot the spatial distribution

# Fix Column Names
# R automatically converts spaces to dots. We replace dots with underscores
names(crime_clean) <- gsub("\\.", "_", names(crime_clean))

# Load Map File
all_files <- list.files("/kaggle/input", recursive = TRUE, full.names = TRUE)
map_file  <- all_files[grep("geojson", all_files, ignore.case = TRUE)][1]

print(paste("Loading map from:", map_file))
chicago_map <- st_read(map_file, quiet = TRUE)

# Remove rows with missing community area
valid_crimes <- subset(crime_clean, !is.na(community_area))

# Create a count table
crime_table  <- table(valid_crimes$community_area)

# Convert table to data frame for merging
crime_counts <- as.data.frame(crime_table)
names(crime_counts) <- c("community_area", "total_crimes")

# Ensure ID is character for matching
chicago_map$area_id <- as.character(chicago_map$area_num_1)

# Merge map with crime data (all.x=TRUE keeps areas with 0 crimes)
map_data <- merge(chicago_map, crime_counts, 
                  by.x = "area_id", 
                  by.y = "community_area", 
                  all.x = TRUE)


# Plot
plot(map_data["total_crimes"],
     main = "Spatial Distribution of Chicago Crime",
     pal = function(n) rev(heat.colors(n)),  # Reversed heat colors
     nbreaks = 12,        # Number of breaks
     key.pos = 1)        # Legend at the bottom

In [ ]:
# Spatial distribution Between 2012 and 2016


# Filter and count
sub_data <- subset(crime_clean, year %in% c(2012, 2016) & !is.na(community_area))

# Matrix: wows = Community Areas ; Columns = Years
crime_matrix <- table(sub_data$community_area, sub_data$year)

# Convert matrix to data frame for merging
crime_counts <- as.data.frame.matrix(crime_matrix)

# Rename columns to avoid numbers-as-names (e.g., "2012" -> "crime_2012")
colnames(crime_counts) <- paste0("crime_", colnames(crime_counts))

# Add the ID column from the row names
crime_counts$community_area <- rownames(crime_counts)

# Merge with Map
chicago_map$area_id <- as.character(chicago_map$area_num_1)
map_comparison <- merge(chicago_map, crime_counts, 
                        by.x = "area_id", 
                        by.y = "community_area", 
                        all.x = TRUE)

# Fill NAs with 0 (for areas that had 0 crimes)
map_comparison$crime_2012[is.na(map_comparison$crime_2012)] <- 0
map_comparison$crime_2016[is.na(map_comparison$crime_2016)] <- 0


# Convert to "Long" format so ggplot can split by Year
map_long <- pivot_longer(map_comparison, 
                         cols = c("crime_2012", "crime_2016"),
                         names_to = "Year", 
                         values_to = "Count")

# Clean the Year labels (remove "crime_" prefix)
map_long$Year <- gsub("crime_", "", map_long$Year)

# Plot side-by-side
final_map <- ggplot(map_long) +
  geom_sf(aes(fill = Count), lwd = 0.1) +       # lwd adjusts border thickness
  scale_fill_gradientn(colors = rev(heat.colors(10)),
                       name = "Count") +  # Legend Title
  facet_wrap(~Year) +                           # Splits into 2 panels
  theme_void() +                                # Removes axis/grid
  theme(legend.position = "right",
        strip.text = element_text(size = 14, face = "bold")) # Header styling

ggsave("result_one_2.png", plot = final_map, width = 12, height = 6, dpi = 300)
final_map

In [ ]:
# Pearson correlation analysis 


# Get the correlation number
cor_value <- cor(map_comparison$crime_2012, map_comparison$crime_2016, method = "pearson")
print(cor_value)

# Create the plot with statistics

ggscatter(map_comparison, x = "crime_2012", y = "crime_2016",
          add = "reg.line",      # Add regression line
          conf.int = TRUE,       # Add confidence interval (shaded area)
          cor.coef = TRUE,       # Show correlation coefficient & p-value
          cor.method = "pearson",
          xlab = "Crime 2012", ylab = "Crime 2016")

# **3. Spatial variations across distinct crime categories**

In [ ]:
# Spatial distribution across 4 categories


# Setup the data (as before)
top_crimes <- c("THEFT", "BATTERY", "CRIMINAL DAMAGE", "NARCOTICS")

crime_types_data <- crime_clean %>%
  filter(year == 2016) %>%
  filter(primary_type %in% top_crimes) %>%
  filter(!is.na(community_area)) %>%
  count(community_area, primary_type, name = "count") %>%
  mutate(community_area = as.character(community_area))

map_types <- chicago_map %>%
  mutate(area_id = as.character(area_num_1)) %>%
  left_join(crime_types_data, by = c("area_id" = "community_area"))

# Plot
options(repr.plot.width = 12, repr.plot.height = 10)

p <-ggplot(map_types) +
  geom_sf(aes(fill = count), color = "white", size = 0.1) +
  scale_fill_gradient(low = "#ffcccc", high = "#990000", na.value = "grey90") +
  facet_wrap(~primary_type) +
  labs(
    fill = "Incidents"
  ) +
  theme_void() +
  theme(
    plot.title = element_text(size = 20, face = "bold", hjust = 0.5),
    strip.text = element_text(size = 15, face = "bold")
  )
p

ggsave("crime_heatmap_2016.png", plot = p, width = 12, height = 10, dpi = 300)

In [ ]:
# Chi-squared test to analyse distribution


# Create count table
chisq_data <- crime_clean %>%
  filter(year == 2016) %>%
  filter(primary_type %in% top_crimes)

# Pearson's X2 test
chisq_result <- chisq.test(table(chisq_data$primary_type, chisq_data$community_area))

print(chisq_result)

# **4. ARIMA time-series model for prediction**

In [ ]:
# Prepare Data (Weekly Counts)
weekly_bins <- cut(crime_clean$date_time, breaks = "week")
weekly_table <- table(weekly_bins)

# Convert to Data Frame to handle the dates
df_weekly <- as.data.frame(weekly_table)
names(df_weekly) <- c("date_str", "count")

# Convert the factor date strings back to Date objects
df_weekly$date <- as.Date(df_weekly$date_str)

# Filter for years 2012-2016
df_weekly$year <- as.numeric(format(df_weekly$date, "%Y"))
df_clean <- subset(df_weekly, year >= 2012 & year <= 2016)

# Split Data
train_data <- subset(df_clean, year < 2016)   # Training set
test_data  <- subset(df_clean, year == 2016)  # Testing set

# Create Time Series
# Frequency = 52 weeks per year
ts_train <- ts(train_data$count, frequency = 52)
# Start the test series at Year 5 (since train was 4 years long)
ts_test  <- ts(test_data$count, frequency = 52, start = c(5, 1))

# Train & Forecast
fit_arima <- auto.arima(ts_train)
forecast_result <- forecast(fit_arima, h = 52)

# Visualize
plot(forecast_result, 
     main = "Can we predict crime?",
     xlab = "Time (Weeks)",
     ylab = "Weekly Crime Count",
     flty = 2) # Dashed line for the forecast

# Add the Actual 2016 Data (Red Line)
lines(ts_test, col = "red", lwd = 2)

# Add a standard Legend
legend("bottomleft", 
       legend = c("ARIMA Forecast", "Actual 2016 Data"),
       col = c("blue", "red"), 
       lty = 1, lwd = 2, bty = "n")

In [ ]:
# Calculate the Mean Absolute Percentage Error (MAPE)


# Calculate accuracy metrics against the tested data (2016)
acc_metrics <- accuracy(forecast_result, ts_test)

# View the full table (Training set vs Test set results)
print(acc_metrics)

# MAPE test
mape_val <- acc_metrics["Test set", "MAPE"]

cat("MAPE:", round(mape_val, 2), "%")